# OGB — OrbitalGuard: YOLOv8n Training
**IBM Bob AI Builders Challenge — August 2025 | Theme: Advance Space Exploration with AI**

Trains `yolov8n` on the Space Debris v2 dataset (11 classes, 2,467 images).  
At the end, writes `metrics.json` and saves `best.pt` to Google Drive.

> ⚠️ **Before running:** Set *Runtime → Change runtime type → T4 GPU*

**Dataset:** Space Debris v2 — woah-noah / Roboflow Universe — CC BY 4.0  
https://universe.roboflow.com/woah-noah/space-debris-mugw2/dataset/2

In [ ]:
# ── 0. Install ─────────────────────────────────────────────────────────────
!pip install -q ultralytics roboflow pyyaml
import torch
print(f'PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU detected — training will be very slow on CPU')

In [ ]:
# ── 1. Mount Google Drive ──────────────────────────────────────────────────
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE_OUTPUT = '/content/drive/MyDrive/ogb_weights'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f'Output dir: {DRIVE_OUTPUT}')

In [ ]:
from google.colab import files
uploaded = files.upload()  # pilih Space_debris_v2i_yolov8.zip

import zipfile, os
with zipfile.ZipFile(list(uploaded.keys())[0], 'r') as z:
    z.extractall('space-debris-v2')

   # cek struktur hasil extract dulu sebelum lanjut
for root, dirs, fnames in os.walk('space-debris-v2'):
    if 'data.yaml' in fnames:
        print('data.yaml ditemukan di:', root)

DATA_YAML = 'space-debris-v2/data.yaml'
print('Dataset root :', 'space-debris-v2')

In [ ]:
# ── 3. Sanity check ────────────────────────────────────────────────────────
# Abort immediately on any mismatch rather than wasting a training run.
import yaml
from pathlib import Path

EXPECTED_CLASSES = [
    'cheops', 'debris', 'double_start', 'earth_observation_sat_1',
    'lisa_pathfinder', 'proba_2', 'proba_3_csc', 'proba_3_ocs',
    'smart_1', 'soho', 'xmm_newton'
]  # double_start — not double_star
EXPECTED_SPLITS = {'train': 2105, 'valid': 239, 'test': 123}

with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)

assert cfg.get('nc') == 11, f'nc={cfg.get("nc")}, expected 11'
assert cfg['names'] == EXPECTED_CLASSES, (
    f'Class mismatch!\nExpected: {EXPECTED_CLASSES}\nGot: {cfg["names"]}'
)
print(f'[OK] {cfg["nc"]} classes verified')

base = Path(DATA_YAML).parent
for split, expected in EXPECTED_SPLITS.items():
    folder = base / split / 'images'
    count = len(list(folder.glob('*'))) if folder.exists() else 0
    status = 'OK' if count == expected else 'WARN'
    print(f'[{status}] {split}: {count} images (expected {expected})')

print('\nSanity check complete — proceeding to training.')

In [ ]:
# ── 4. Train YOLOv8n ───────────────────────────────────────────────────────
# Hyperparameters locked per OGB spec.
# fliplr/flipud disabled: dataset already contains Roboflow-applied flips.
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    optimizer='AdamW',
    device=0,
    workers=2,
    project='/content/runs',
    name='ogb_yolov8n',
    fliplr=0.0,   # already flipped in dataset
    flipud=0.0,   # already flipped in dataset
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    mosaic=1.0,
    mixup=0.0,
    val=True,
    plots=True,
    save=True,
)

print(f'\nRun saved to: {results.save_dir}')

In [ ]:
# ── 5. Evaluate on test split ──────────────────────────────────────────────
from pathlib import Path

best_pt = Path(results.save_dir) / 'weights' / 'best.pt'
eval_model = YOLO(str(best_pt))

test_metrics = eval_model.val(
    data=DATA_YAML,
    split='test',
    imgsz=640,
    batch=16,
    device=0,
)

m = test_metrics.results_dict
map50   = m.get('metrics/mAP50(B)',    float('nan'))
map5095 = m.get('metrics/mAP50-95(B)', float('nan'))
prec    = m.get('metrics/precision(B)', float('nan'))
rec     = m.get('metrics/recall(B)',    float('nan'))
f1      = 2*prec*rec/(prec+rec) if (prec+rec) > 0 else float('nan')

print('\n' + '='*55)
print('OGB — Test set evaluation')
print('='*55)
print(f'  mAP@50      : {map50:.4f}   (target >= 0.85)')
print(f'  mAP@50-95   : {map5095:.4f}')
print(f'  Precision   : {prec:.4f}')
print(f'  Recall      : {rec:.4f}')
print(f'  F1          : {f1:.4f}')
print('='*55)

In [ ]:
# ── 6. CPU inference latency benchmark ─────────────────────────────────────
# Deployment target is CPU — measure on CPU even though training was GPU.
import time, numpy as np

cpu_model = YOLO(str(best_pt))
dummy = np.zeros((640, 640, 3), dtype=np.uint8)

# Warm-up (3 runs)
for _ in range(3):
    cpu_model.predict(dummy, imgsz=640, device='cpu', verbose=False)

# Timed runs
N = 20
times = []
for _ in range(N):
    t0 = time.perf_counter()
    cpu_model.predict(dummy, imgsz=640, device='cpu', verbose=False)
    times.append((time.perf_counter() - t0) * 1000)

lat_mean   = float(np.mean(times))
lat_median = float(np.median(times))
lat_p95    = float(np.percentile(times, 95))

print(f'CPU inference latency ({N} runs):')
print(f'  Mean   : {lat_mean:.1f} ms')
print(f'  Median : {lat_median:.1f} ms')
print(f'  p95    : {lat_p95:.1f} ms')

In [ ]:
# ── 7. Export metrics.json + copy weights to Drive ─────────────────────────
# This file is read by update_readme.py to fill in the README automatically.
import json, shutil

metrics_out = {
    'map50':            round(map50,   4),
    'map5095':          round(map5095, 4),
    'precision':        round(prec,    4),
    'recall':           round(rec,     4),
    'f1':               round(f1,      4),
    'cpu_latency_mean_ms':   round(lat_mean,   1),
    'cpu_latency_median_ms': round(lat_median, 1),
    'cpu_latency_p95_ms':    round(lat_p95,    1),
    'epochs': 50,
    'imgsz':  640,
    'batch':  16,
}

metrics_path = f'{DRIVE_OUTPUT}/metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics_out, f, indent=2)
print(f'metrics.json saved to: {metrics_path}')

weights_dest = f'{DRIVE_OUTPUT}/ogb_yolov8n.pt'
shutil.copy2(str(best_pt), weights_dest)
print(f'Weights saved to: {weights_dest}')

print('\n--- Next steps ---')
print('1. Download ogb_yolov8n.pt  →  place at  ml/weights/ogb_yolov8n.pt')
print('2. Download metrics.json    →  place at  ml/weights/metrics.json')
print('3. Run: python ml/training/update_readme.py')
print('4. Run: python ml/inference/smoke_test.py')